# 01a – Khảo sát chất lượng dữ liệu ER (EDA)

Chỉ **đọc** `data/raw/Hospital_Emergency_Room_Data_Raw.csv`, không ghi file nào (chạy EDA không đụng tầng Silver). Kết quả là cơ sở cho mục 3.2 và 3.3.1 của báo cáo và cho quy tắc làm sạch trong `02a`.

Danh sách kiểm tra: số dòng, cột, khoảng thời gian, có giờ vào hay không, tỷ lệ thiếu điểm hài lòng, nhãn `None` của khoa chuyển tuyến, nhãn giới tính, cột lặp, phân phối thời gian chờ theo giờ.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                            if (p / 'src' / 'anon_common.py').exists()) / 'src'))
import numpy as np
import pandas as pd
from anon_common import RAW
pd.set_option('display.width', 160)

## 1. Cấu trúc và thang đo

In [ ]:
# Quan trọng: "None" ở Department Referral là một nhãn, không phải thiếu dữ liệu.
er = pd.read_csv(RAW / 'Hospital_Emergency_Room_Data_Raw.csv', keep_default_na=False, na_values=[''])
er_default = pd.read_csv(RAW / 'Hospital_Emergency_Room_Data_Raw.csv')            # cách đọc mặc định của pandas
print('Số dòng, số cột:', er.shape)
print('Dòng mất nếu đọc mặc định (None bị coi là thiếu):', int(er_default['Department Referral'].isna().sum()))

scale = {'Patient Id': 'Nominal (mã định danh)', 'Patient Admission Date': 'Interval (ngày giờ)',
         'Patient First Inital': 'Nominal (định danh)', 'Patient Last Name': 'Nominal (định danh)',
         'Patient Gender': 'Nominal', 'Patient Age': 'Ratio (năm)', 'Patient Race': 'Nominal',
         'Department Referral': 'Nominal', 'Patient Admission Flag': 'Nominal (nhị phân)',
         'Patient Satisfaction Score': 'Ordinal, thang 0-10', 'Patient Waittime': 'Ratio (phút)',
         'Patient Admission Flag.1': 'Trùng lặp'}
profile = pd.DataFrame({'dtype': er.dtypes.astype(str), 'n_missing': er.isna().sum(), 'n_unique': er.nunique(),
                        'thang_do': pd.Series(scale)})
profile['pct_missing'] = (100 * profile.n_missing / len(er)).round(1)
profile

## 2. Thời gian và giờ vào

In [ ]:
dt = pd.to_datetime(er['Patient Admission Date'], format='%d-%m-%Y %H:%M')
print('Từ', dt.min(), 'đến', dt.max(), '| số tháng:', dt.dt.to_period('M').nunique())
print('Có giờ vào chính xác đến phút:', dt.dt.minute.nunique() > 1, '| số giờ khác nhau trong ngày:', dt.dt.hour.nunique())
print('Mốc thời gian trùng nhau:', int(dt.duplicated().sum()))
print('Mã bệnh nhân trùng:', int(er['Patient Id'].duplicated().sum()), '| dòng trùng toàn bộ:', int(er.duplicated().sum()))
print('Patient Id khớp mẫu ###-##-####:', er['Patient Id'].str.match(r'^\d{3}-\d{2}-\d{4}$').mean())

## 3. Điểm hài lòng, khoa chuyển tuyến, giới tính, cột lặp

In [ ]:
sat = er['Patient Satisfaction Score']
print('Thiếu điểm hài lòng:', int(sat.isna().sum()), f'({100 * sat.isna().mean():.1f}%) | có điểm:', int(sat.notna().sum()), '| trung bình:', round(sat.mean(), 2))
print('Tỷ lệ thiếu theo nhập viện:\n', er.groupby('Patient Admission Flag')['Patient Satisfaction Score'].apply(lambda s: round(100 * s.isna().mean(), 1)).to_string())
print('\nKhoa chuyển tuyến:\n', er['Department Referral'].value_counts().to_string())
print('\nGiới tính (nhãn gốc):', er['Patient Gender'].value_counts().to_dict())
print('Hai cột nhập viện giống hệt nhau:', bool((er['Patient Admission Flag'] == er['Patient Admission Flag.1']).all()))

## 4. Thời gian chờ theo giờ, thứ, khoa

In [ ]:
w = er['Patient Waittime']
print(w.describe().round(2).to_string())
print('Chờ < 30 phút:', round(100 * (w < 30).mean(), 1), '% | chờ <= 30 phút:', round(100 * (w <= 30).mean(), 1), '%')
by_hour = er.groupby(dt.dt.hour)['Patient Waittime'].agg(['count', 'mean']).round(1)
print('\nTheo giờ: trung bình nhỏ nhất/lớn nhất =', by_hour['mean'].min(), '/', by_hour['mean'].max(),
      '| 8h, 9h, 10h =', by_hour.loc[[8, 9, 10], 'mean'].tolist(), '| số lượt mỗi giờ =', by_hour['count'].min(), '-', by_hour['count'].max())
print('Theo thứ trong tuần (0 = Thứ Hai):', er.groupby(dt.dt.dayofweek)['Patient Waittime'].mean().round(1).to_dict())
print('Theo khoa:', er.groupby('Department Referral')['Patient Waittime'].mean().round(1).to_dict())
print('Tương quan chờ - hài lòng:', round(w.corr(sat), 3))
by_hour.T

## 5. Danh sách vấn đề chất lượng (có số liệu)

| Vấn đề | Số liệu | Xử lý ở `02a` |
|---|---|---|
| Điểm hài lòng thiếu | 6.699/9.216 (72,7%) | Giữ rỗng, ghi cỡ mẫu |
| Nhãn `None` bị pandas đọc thành thiếu | 5.400 dòng (58,6%) | Đọc với `keep_default_na=False`, đổi thành `No referral` |
| Nhãn giới tính không thống nhất | M 4.700, F 4.470, Male 29, Female 17 | Chuẩn hóa M/F |
| Cột lặp | `Patient Admission Flag.1` | Bỏ |
| Thời gian chờ phẳng theo giờ | trung bình theo giờ 33,9 đến 37,2 phút | Không có đỉnh 8h-10h; trình bày như kịch bản giả định |